In [53]:
import json
from pathlib import Path
import pandas as pd

# Read results
- Parse merged results for different meshsizes
- Export as latex table

In [65]:
# form df with restuls from multiple runs
results_list = []
for file in Path("outputs").glob("*/results.json"):
    with open(file, 'r') as f:
        results = json.load(f)
        results_list.append(results)
df = pd.DataFrame(results_list)
df.sort_values(by="mesh_shape", inplace=True) # So they go from less to more accurate
df

,file_name,mesh_shape,num_iterations,k,peak_to_avg,max_power_density,run_time
2,ANLB_9x9x19,"[9, 9, 19]",357,1.030256,2.554313,27050.589422,11.318578
1,ANLB_18x18x19,"[18, 18, 19]",347,1.027463,2.662420,7048.865056,97.956389
0,ANLB_36x36x38,"[36, 36, 38]",351,1.027075,2.699035,893.225670,9815.401780


In [66]:
# only keep columns we care about
df.drop(columns=["file_name", "max_power_density"], inplace=True)

# convert run time to hrs:mins:secs
df["run_time"] = pd.to_timedelta(df["run_time"], unit="s").apply(
    lambda x: f"{int(x.total_seconds()//3600):02}:{int((x.total_seconds()%3600)//60):02}:{int(x.total_seconds()%60):04.1f}"
)

# fix names to print nicely in latex table
df = df.rename(columns={
    "mesh_shape": "Mesh Shape",
    "num_iterations": "Num Iterations",
    "final_k": "k eff",
    "peak_to_avg": "Peak to Average Power Density",
    "run_time": "Run Time (hr:m:s)"
})

# print df as latex table
latex_str = df.to_latex(
    index=False,
    escape=True,
    column_format="|c|c|c|c|c|",
)

# Replace booktabs rules with \hline to match your template style
latex_str = latex_str.replace(r"\toprule", r"\hline")
latex_str = latex_str.replace(r"\midrule", r"\hline")
latex_str = latex_str.replace(r"\bottomrule", r"\hline")
latex_str = latex_str.replace(r"\\", r"\\ \hline")  # adds \hline after every row

with open("FDM_results_table.tex", 'w') as f:
    f.write(latex_str)